# Kuvantunnistus omilla kuvilla - CNN

Datan ymmärrys
Datasetti sisältää yhtensä 250 kuvaa ja kuusi luokkaa. Kuvien luokat ja niiden kuvamäärät ovat seuraavat:
Omena: 50 kuvaa
Avokaado: 50 kuvaa
Banaani: 50 kuvaa
Viinirypäle: 50 kuvaa
Kiivi: 50 kuvaa
Appelsiini: 50 kuvaa

## Datan esikäsittely
Kuvadata on jaettu valmiiksi 70%, 15%, 15% jaolla testi- validointi- ja koulutushakemistoon. Valmiin hakemistorakenteen avulla voidaan ladata data suoraan kolmeksi Dataset-olioksi `tf.keras.preprocessing.image_dataset_from_directory`-funktiolla. Funktion parametreina annetaan hakemistojen polku, kuvien koko, eräkoko, labelien muoto ja sekoitus, jolloin data kuvat sekoitetaan.  Eräkoko määritetään jo tässä vaiheessa, jolloin datasetti palauttaa erissä 32 kuvaa ja niiden vastaavat luokat. Tämän takia eräkokoa ei tarvitse määrittää enää mallin koulutuksen yhteydessä. 

Datasetti normalisoidaan jakamalla kuvat 255:llä. Datasetti jaetaan koulutus-, validointi- ja testidatasetteihin 70%, 15% ja 15% suhteessa.

# Oma CNN-malli

Ladataan kuvat 224x224 kokoisina ja jaetaan ne kouluts-validointi ja testihakemistoon. Normalisoidaan kuvat jakamalla ne 255:llä


In [ ]:
import keras.src.callbacks
import tensorflow as tf
import numpy as np
from numpy.f2py.crackfortran import verbose
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, RandomFlip, RandomRotation, RandomZoom
import random

batch_size = 32
img_size = (224, 224)

# Ladataan datasetti
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "../datasets/kuvantunnistus/train",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=True,
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "../datasets/kuvantunnistus/validation",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=True,
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "../datasets/kuvantunnistus/test",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_dataset.class_names

# Normalisoidaan kuvat
train_dataset = train_dataset.map(lambda x, y: (x / 255.0, y))
validation_dataset = validation_dataset.map(lambda x, y: (x / 255.0, y))
test_dataset = test_dataset.map(lambda x, y: (x / 255.0, y))

# class names

# Tarkistetaan datasetin koko
print(f"Train batches: {len(train_dataset)}, Validation batches: {len(validation_dataset)}, Test batches: {len(test_dataset)}")

## CNN-mallin rakenne
### Syötekerros
Mallin syötekerroksena on konvoluutioverkko, joka ottaa syötteenä värikuvia kooltaan 224x224. Syötekerroksessa on 32 filtteriä ja jokainen filtteri koostuu 3x3 painokertoimesta. Aktivaatiofunktiona käytetään ReLua.
### Piilokerrokset
Mallissa käytetään datan augmentaatiota, jotta dataa olisi riittävästi ja se olisi tarpeeksi monipuolista. MaxPoolingia käytetään tiivistämään kuvasta opittavaa tietoa. Piilokerroksissa on yksi konvoluutiokerros jossa on 64 suodatinta samoilla painokerroksilla sekä aktivaatiofunktiolla kuin syötekerroksessa, flatten-kerros, tiheä-kerros 128:lla neuronilla ja ReLu aktivaatiofunktiolla ja dropout kerros, jossa sammutetaan puolet neuroneista.
### Ulostulokerros
Ulostulokerroksena on tiheä-kerros 6 neuronilla. Aktivaatiofunktiona käytetään softmaxia, jotta ulostulevat ennusteet ovat kaikki 0-1 välillä ja niiden summa olisi 1.
### Takaisinkutsufunktiot
Mallissa käytetään EarlyStoppingia kärsivällisyydellä 5 ja parhaat painoarvota palautetaan. EarlyStopping monitoroi validoinnin häviötä.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.2),
])

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    data_augmentation,
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    #ReduceLROnPlateau(factor=0.1, patience=3)
]

model.summary()
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=100, callbacks=callbacks, verbose=0)

In [ ]:
import matplotlib.pyplot as plt


def plot_loss(history):
    plt.plot(history.history["loss"], label="Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.legend()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid()
    plt.show()

plot_loss(history)

In [ ]:
def print_test_loss_accuracy(model):
    score = model.evaluate(test_dataset)
    print('Test loss:', score[0])
    print('Test accuracy:', score[1])

print_test_loss_accuracy(model)

# CNN-mallin arviointi
Mallin tarkkuus on noin 0.50 ja häviö 1.5. Tulosten perusteella malli on liian yksinkertainen, eikä se opi tarpeeksi hyvin.

In [ ]:
# Haetaan testidatasetista yksi batch
test_images, test_labels = next(iter(test_dataset.take(1)))

# Ennustetaan mallilla
predictions = model.predict(test_images)

# Muunnetaan one-hot-enkoodatut labelit kokonaisluvuiksi
true_labels = np.argmax(test_labels, axis=1)
predicted_labels = np.argmax(predictions, axis=1)


def plot_images(model):
    y_pred = []
    x_images = []
    y_true = []

    for images, labels in test_dataset:
        y_pred.extend(model.predict(images, batch_size=32))
        x_images.extend(images.numpy())
        y_true.extend(labels.numpy())

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    x_images = np.array(x_images)

    num_samples = 5
    random_indices = random.sample(range(len(x_images)), num_samples)

    for i in random_indices:
        plt.figure(figsize=(10, 4))

        # Display image
        plt.subplot(1, 2, 1)
        plt.imshow((x_images[i] * 255).astype("uint8"))
        true_class = class_names[np.argmax(y_true[i])]
        plt.title(f"True: {true_class}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        pred_prob = y_pred[i]
        predicted_class = class_names[np.argmax(pred_prob)]

        colors = ['lightgreen' if (class_names[idx] == true_class) else 'skyblue'
                  for idx in range(len(class_names))]

        plt.barh(class_names, pred_prob, color=colors)
        plt.xlabel('Probability')
        plt.title(f"Predicted: {predicted_class}")
        plt.xlim([0, 1])

        if predicted_class != true_class:
            plt.text(0.5, -0.5, "Incorrect", color='red',
                     ha='center', transform=plt.gca().transAxes)
        else:
            plt.text(0.5, -0.5, "Correct", color='green',
                     ha='center', transform=plt.gca().transAxes)

        plt.tight_layout()
        plt.show()


plot_images(model)

# Kuvantunnistus omilla kuvilla – VGG16 Assisted

Hyödynnetään VGG16-mallia, joka on esikoulutettu ImageNet-datasetillä. Viimeinen kerros poistetaan, jotta malli voidaan mukauttaa omaan datasettiin. Mallin esikoulutetut kerrokset jäädytetään, jotta niitä ei kouluteta uudelleen, ja niiden päälle lisätään oma mukautettu osuus.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.regularizers import l2

vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg16_base.trainable = False

## Mallin rakenteen määrittely
## Syötekerros
Mallin pohjana käytetään esikoulutettua VGG16-verkkoa, jonka yläosaa laajennetaan omilla kerroksilla. Syötekuvat ovat kooltaan 224x224 ja sisältävät kolme väriulottuvuutta (RGB).
## Piilokerrokset
VGG16:n valmiin piirrekaartan päälle lisätään ensin GlobalAveragePooling2D-kerros, joka tiivistää piirteet yhdelle vektorille säilyttäen olennaisen informaation. Sen jälkeen verkkoon lisätään kaksi tiheää (Dense) kerrosta, ensin 256 neuronilla ja sitten 128 neuronilla, joista jälkimmäiseen on liitetty L2-regularisointi (arvolla 0.01) ylisovittamisen vähentämiseksi. Tämän jälkeen lisätään kolmas tiheä kerros,myös 128 neuronilla ja ReLU-aktivoinnilla. Dropout-kerros (0.2) sammuttaa satunnaisesti 20 % neuroneista koulutuksen aikana, mikä parantaa mallin yleistettävyyttä.
## Ulostulokerros
Ulostulokerroksena toimii Dense-kerros, jossa on 6 neuronia ja aktivointina softmax. Tämä mahdollistaa mallin antavan todennäköisyysjakauman kuudelle eri luokalle, ja ennusteiden summan olevan yksi.
### Takaisinkutsufunktiot
Mallissa käyetään ensimmäisen mallin luonnin yhteydessä luotuja takaisinkutsufunktioita, eli EarlyStoppingia kärsivällisyydellä 5. Myös tässäkin, parhaat arvot palautetaan.

In [ ]:
x = vgg16_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(6, activation='softmax')(x)

## Mallin rakentaminen ja optimointi

Malli koostuu VGG16-pohjasta ja uusista lisäkerroksista. Optimointimenetelmänä käytetään Adam-algoritmia, kustannusfunktio on categorical_crossentropy, ja suorituskykymittarina käytetään tarkkuutta (accuracy).

In [ ]:
model2 = Model(inputs=vgg16_base.input, outputs=predictions)

model2.compile(optimizer='adam',
               loss='categorical_crossentropy',
               metrics=['accuracy'])

history2 = model2.fit(train_dataset, validation_data=validation_dataset, callbacks=callbacks, epochs=100, verbose=0)

In [ ]:
plot_loss(history2)

## Mallin suorituskyvyn arviointi

Esikoulutetun VGG16-mallin tarkkuus on noin 88 %, ja kustannusfunktio noin 0.86.

# Esikoulutetun ja oman mallin vertailu
Mallit koulutettiin 100 epookilla ja eräkoolla 32. Ylioppimisen vähentämiseksi käytettiin EarlyStopping-callbackia, jonka kärsivällisyysarvo on 5. Tämä tarkoittaa, että koulutus keskeytyy, jos validointihäviö ei parane viiden epookin aikana, ja samalla palautetaan parhaat painotukset. Lisäksi ReduceLROnPlateau-callback pienentää oppimisnopeutta, jos validointihäviö ei parane kolmen epookin aikana.

Oma CNN-malli saavutti oppimisen noin 20 epookin kohdalla, kun taas esikoulutettu VGG16-malli tarvitsi noin 40 epookkia. Esikoulutetun mallin tarkkuus on 80 %, mikä on hieman parempi kuin oman mallin noin 50% tarkkuus. Suurempi ero näkyy kuitenkin kustannusfunktiossa: VGG16:n kustannusarvo on 0.66, kun taas oman mallin kustannusarvo on 1.5, mikä osoittaa esikoulutetun mallin selvästi paremman suorituskyvyn.

In [ ]:
print_test_loss_accuracy(model2)

In [ ]:
plot_images(model2)

# Hienosäädetty esikoulutettu malli VGG16

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# Load VGG16 base model
vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze initial layers and unfreeze later ones for fine-tuning
for layer in vgg16_base.layers[:15]:  # Freeze first 15 layers
    layer.trainable = False
for layer in vgg16_base.layers[15:]:  # Unfreeze last few layers
    layer.trainable = True

# Add custom layers
x = vgg16_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(6, activation='softmax')(x)

# Create model
model3 = Model(inputs=vgg16_base.input, outputs=predictions)

model3.summary()

optimizer = Adam(learning_rate=0.00001)  # Lower learning rate for fine-tuning

model3.compile(optimizer=optimizer,
               loss='categorical_crossentropy',
               metrics=['accuracy'])

history3 = model3.fit(train_dataset,
                      validation_data=validation_dataset,
                      epochs=100,  # More epochs for fine-tuning
                      callbacks=callbacks)

In [ ]:
plot_loss(history3)

In [ ]:
print_test_loss_accuracy(model3)

In [ ]:
plot_images(model3)